<a href="https://colab.research.google.com/github/Cambie26/industrial_agent/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Industrial agent - demo

An agent that answers maintenance questions about a fleet of 100 turbofan
engines, using tools over the NASA C-MAPSS dataset (subset FD001).

There is no retrieval and no fine-tuning here. The model is given three tools
and decides for itself which to call, how many times, and when it has enough
to answer. Each cell below is one question; the printed output shows every
tool call the model made before answering.

All the logic lives in `src/industrial_agent/` - this notebook only drives it.

## Setup

In [ ]:
import os

if not os.path.exists("industrial_agent"):
    !git clone -q https://github.com/Cambie26/industrial_agent.git
%pip install -q -e ./industrial_agent

The agent needs an Anthropic API key. In Colab, add a secret named
`ANTHROPIC_API_KEY` (key icon in the left sidebar) and grant this notebook
access to it.

In [ ]:
from industrial_agent import default_chat, load_fleet, sensor_traces

chat = default_chat()          # downloads the data, calibrates, wires up tools
print("model:", chat.agent.model)
print("tools:", ", ".join(chat.agent.tools.registry))

## The data

100 engines currently in service. Each row is one flight cycle - takeoff,
cruise, landing - with 21 sensor readings (temperatures, pressures, fan and
core speeds, fuel flow ratio, bleed enthalpy) and the three operating settings
the engine was run at.

Degradation is visible to the eye well before failure, which is what makes the
wear index possible:

In [ ]:
fleet = load_fleet()
print(f"{fleet.unit.nunique()} engines, {len(fleet):,} cycles recorded")
fleet.head()

In [ ]:
sensor_traces(fleet);

## 1. Fleet triage

An open question with no unit number in it. The model has to reach for the
fleet-wide tool rather than asking which engine we mean.

In [ ]:
chat.ask("Which engines need attention?")

## 2. One engine in depth

Now a specific unit. Note the model leads with the judgement and keeps the
sensor detail as supporting evidence - it is not dumping the tool output back
at us.

In [ ]:
chat.ask("How is unit 76 doing?")

## 3. Two engines, one turn

Two independent lookups. The model issues both tool calls in a single turn
rather than going round the loop twice.

In [ ]:
chat.ask("Compare units 76 and 55")

## 4. A follow-up that needs memory

No unit numbers, no new data. To answer this the model has to use what it
already learned in the previous turns - so it should call no tools at all.

In [ ]:
chat.ask("Of those two, which would you ground first, and why?")

## 5. Something the tools cannot answer

The system prompt tells the agent never to invent readings. There is no
hangar-temperature tool, so the honest answer is that it cannot say.

In [ ]:
chat.ask("What is the hangar temperature?")

## 6. A unit that does not exist

The fleet is units 1-100. The tool returns a plain-language message rather
than raising, so the model can recover and say something useful.

In [ ]:
chat.ask("Give me the readings for unit 250")

## 7. A question that needs several steps

Ranking is one tool call; deciding what to do about the top engines needs
another look at each. This is where the loop earns its keep.

In [ ]:
chat.reset()   # fresh conversation, so this stands on its own
chat.ask("Of the three most worn engines, which shows the clearest "
         "high-pressure turbine problem? Explain what you looked at.")

## Save the transcript

Everything above, written out as markdown so it can be read straight from the
repo without opening the notebook.

In [ ]:
chat.save("demo_transcript.md")